In [ ]:
!pip install torch_geometric
!pip install adamp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for adamp: filename=adamp-0.3.0-py3-none-any.whl size=5984 sha256=4b2a6885b1d273558d9be8355a66d3ab31bbabf66c1eef9dd412fa282f658ef4
  Stored in directory: /root/.cache/pip/wheels/c7/ad/0f/b41b1c45b18c66e5eef5d2254415af8055c7e2b0934145157d
Successfully built adamp


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv, BatchNorm
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.nn import Dropout, Linear, GRU
import numpy as np
import random

# Set random seeds for reproducibility
def set_random_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Define the GRNN model with additional layers for greater complexity
class GRNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super(GRNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.norm1 = BatchNorm(hidden_channels)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.norm2 = BatchNorm(hidden_channels)

        # Adding an extra GCN + GRU layer
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.norm3 = BatchNorm(hidden_channels)
        self.gru = GRU(hidden_channels, hidden_channels)

        self.conv4 = GCNConv(hidden_channels, hidden_channels)
        self.norm4 = BatchNorm(hidden_channels)

        self.fc = Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Apply GCN layers followed by GRU for sequential processing
        x = F.relu(self.norm1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.norm2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Sequential processing with GRU
        x, _ = self.gru(x.unsqueeze(0))
        x = x.squeeze(0)

        x = F.relu(self.norm3(self.conv3(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = global_mean_pool(x, batch)

        x = self.fc(F.dropout(x, p=self.dropout, training=self.training))
        return F.log_softmax(x, dim=1)

# Training function
def train_model(model, loader, epochs=300, lr=0.001, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data in loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

    return model

# Evaluation function
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            out = model(data)
            preds = out.argmax(dim=1)

            if data.y.ndim == 2:
                labels = data.y.argmax(dim=1)
            else:
                labels = data.y

            all_preds.append(out)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0).view(-1)

    out_prob = F.softmax(all_preds, dim=1).cpu()

    current_num_classes = out_prob.shape[1]
    if current_num_classes != num_classes:
        if current_num_classes < num_classes:
            padding = num_classes - current_num_classes
            out_prob = torch.cat([out_prob, torch.zeros(out_prob.size(0), padding)], dim=1)
        else:
            out_prob = out_prob[:, :num_classes]

    auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob[:, 1], average='weighted') if num_classes == 2 else roc_auc_score(all_labels.cpu().numpy(), out_prob, multi_class='ovr', average='weighted')
    logloss = log_loss(all_labels.cpu().numpy(), out_prob)

    test_acc = (all_preds.argmax(dim=1) == all_labels).sum().item() / len(all_labels)
    f1 = f1_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    precision = precision_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    recall = recall_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')

    return {
        'accuracy': test_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc_roc': auc_roc,
        'log_loss': logloss
    }

# Set random seed
set_random_seed(42)

# Load and augment datasets
dataset_proteins = TUDataset(root='/content/PROTEINS', name='PROTEINS', transform=NormalizeFeatures())
dataset_mutag = TUDataset(root='/content/MUTAG', name='MUTAG', transform=NormalizeFeatures())

# Split datasets into training and testing sets (70/30)
train_idx_proteins, test_idx_proteins = train_test_split(range(len(dataset_proteins)), test_size=0.3, random_state=42)
train_dataset_proteins = dataset_proteins[train_idx_proteins]
test_dataset_proteins = dataset_proteins[test_idx_proteins]

train_idx_mutag, test_idx_mutag = train_test_split(range(len(dataset_mutag)), test_size=0.3, random_state=42)
train_dataset_mutag = dataset_mutag[train_idx_mutag]
test_dataset_mutag = dataset_mutag[test_idx_mutag]

train_loader_proteins = DataLoader(train_dataset_proteins, batch_size=32, shuffle=True)
test_loader_proteins = DataLoader(test_dataset_proteins, batch_size=32, shuffle=False)

train_loader_mutag = DataLoader(train_dataset_mutag, batch_size=32, shuffle=True)
test_loader_mutag = DataLoader(test_dataset_mutag, batch_size=32, shuffle=False)

# Initialize and train model on PROTEINS
model_proteins = GRNN(in_channels=dataset_proteins.num_node_features, hidden_channels=128, out_channels=dataset_proteins.num_classes, dropout=0.5)
train_model(model_proteins, train_loader_proteins, epochs=300)

# Evaluate on PROTEINS dataset
print("\nResults on PROTEINS Dataset:")
proteins_results = evaluate_model(model_proteins, test_loader_proteins, num_classes=dataset_proteins.num_classes)
print(proteins_results)

# Transfer learning to MUTAG dataset
model_mutag = GRNN(in_channels=dataset_mutag.num_node_features, hidden_channels=128, out_channels=dataset_mutag.num_classes, dropout=0.5)
state_dict_proteins = model_proteins.state_dict()
state_dict_mutag = model_mutag.state_dict()

for key in state_dict_proteins:
    if key in state_dict_mutag and state_dict_proteins[key].shape == state_dict_mutag[key].shape:
        state_dict_mutag[key] = state_dict_proteins[key]

model_mutag.load_state_dict(state_dict_mutag, strict=False)
train_model(model_mutag, train_loader_mutag, epochs=300)

# Evaluate on MUTAG dataset
print("\nResults on MUTAG Dataset After Transfer Learning:")
mutag_results = evaluate_model(model_mutag, test_loader_mutag, num_classes=dataset_mutag.num_classes)
print(mutag_results)



Results on PROTEINS Dataset:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


{'accuracy': 0.6766467065868264, 'f1': 0.6791933684386402, 'precision': 0.6834689095633045, 'recall': 0.6766467065868264, 'auc_roc': 0.7042887667887667, 'log_loss': 0.6195265761681014}

Results on MUTAG Dataset After Transfer Learning:
{'accuracy': 0.8596491228070176, 'f1': 0.8543003851091142, 'precision': 0.8594159818150027, 'recall': 0.8596491228070176, 'auc_roc': 0.952991452991453, 'log_loss': 0.29844729113517654}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
